# Talk 1. Friday 26 July. 

## Aquí tienes la traducción al español del resumen de la presentación y la biografía del ponente:

---

## Potencia tus habilidades en Python usando el intérprete de Python

* **FORMATO:** Charla
* **NIVEL:** Avanzado
* **IDIOMA:** Inglés

En esta charla, desafiaré la perspectiva de la audiencia sobre Python. Python no es un intérprete y, de hecho, existen múltiples intérpretes de Python, cada uno con su propia arquitectura y propósito. Analizaré a fondo los componentes internos de Python y mostraré cómo interactúan los lenguajes de programación bajo la superficie.

Exploraremos cómo escribir mejor código en Python comprendiendo el recolector de basura (*garbage collector*), qué se puede construir utilizando el AST (Árbol de Sintaxis Abstracta), cómo leer y aprovechar el desensamblador, y las implicaciones prácticas de la transición de Python de su antiguo analizador LL(1) al actual analizador PEG. También nos sumergiremos en características menos conocidas de los intérpretes de Python, qué es realmente un PEP y cómo moldea el lenguaje, y concluiremos con una mirada profunda a Python sin el GIL: qué cambia, qué se rompe y cómo el equipo central (*core team*) lo eliminó.

A lo largo de la charla, compartiré historias personales, incluyendo batallas causadas por ASTs idénticos y el momento en que creí haber descubierto una manera de acelerar el propio intérprete de Python, pero...

---

### Ponente: Daniel Arango Sohm

**Python Junior Software Engineer @ Epam**

> He ganado en 3 ocasiones el premio al mejor proyecto en Ingeniería de Sistemas en EAFIT. He dado charlas en Medellín JS, Python Medellín (en dos ocasiones), Medellín AI, PyCon Colombia, Python Moscow y BarNLP. Soy mitad colombiano, también tengo herencia alemana y rusa, y hablo ruso y alemán.

# Curso de preparación para esta charla  



Como matemático y magister en ciencias matemáticas, tienes una ventaja enorme: tu mente está entrenada para la abstracción extrema, la lógica formal y la teoría de grafos (fundamental para entender compiladores y ASTs).



El libro de Wes McKinney (*Python for Data Analysis*) es fantástico para aprender a "usar" Python para cómputo científico, pero está enfocado en la superficie (la API de pandas, numpy y análisis de datos). La charla de Daniel Arango va en la dirección opuesta: **hacia las profundidades del motor del lenguaje (C Python, intérpretes, compiladores y manejo de memoria).**



Para que no te pierdas de nada y aproveches al máximo tu perfil matemático, he diseñado este plan de estudio estructurado de **5 módulos**. Está pensado para conectar lo que ya sabes con los conceptos de bajo nivel de la charla.

---



## Ruta de Aprendizaje: De la Matemática y los Datos a las Entrañas de Python



### Módulo 1: Desmitificando el Intérprete y la Arquitectura (La Base)

*De la ejecución de celdas en Jupyter a cómo respira la máquina.*

* **El choque cultural:** En Jupyter ejecutas celdas y ves resultados. Pero, ¿qué es realmente "Python"? Aprender la diferencia entre la **especificación del lenguaje** (las reglas de Python escritas en un documento) y sus **implementaciones/intérpretes** (Cpython, PyPy, Jython, IronPython).
* **La anatomía de CPython:** El intérprete estándar (el que usas en Anaconda) está escrito en C. Entender el flujo básico:

$$\text{Código Fuente} \xrightarrow{\text{Parser}} \text{AST} \xrightarrow{\text{Compilador}} \text{Bytecode} \xrightarrow{\text{Máquina Virtual}} \text{Ejecución}$$




* **Análogo Matemático:** Piensa en la especificación de Python como una estructura axiomática, y en los diferentes intérpretes como diferentes modelos que satisfacen esos axiomas.

---



### Módulo 2: Gramáticas y Estructuras (Parsers y AST)

*Donde tu formación matemática brillará con luz propia.*

* **¿Qué es un AST?** El Árbol de Sintaxis Abstracta (*Abstract Syntax Tree*). Es una estructura de datos en forma de **árbol n-ario** que representa la estructura sintáctica del código fuente.
* **De LL(1) a PEG:**
* **LL(1):** Gramáticas libres de contexto analizadas de izquierda a derecha, con derivación por la izquierda y 1 token de anticipación (*lookahead*).
* **PEG (Parsing Expression Grammars):** Un formalismo analítico más potente que evita la ambigüedad y permite a Python tener una sintaxis más flexible desde la versión 3.9.


* **Práctica en VS Code:** Jugar con la librería estándar `ast` en Python.
```python


In [1]:
import ast
node = ast.parse("x = 5 + 3")
print(ast.dump(node, indent=4)) # Verás la estructura de árbol del código


Module(
    body=[
        Assign(
            targets=[
                Name(id='x', ctx=Store())],
            value=BinOp(
                left=Constant(value=5),
                op=Add(),
                right=Constant(value=3)))])



### Módulo 3: El Bytecode y el Desensamblador (El Ensamblador de Python)

*De la comodidad de Jupyter a las instrucciones de la máquina virtual.*

* **¿Qué es el bytecode?** Python no traduce tu código directamente a código máquina (0s y 1s de tu procesador Intel/M1), sino a un lenguaje intermedio de bajo nivel optimizado para la máquina virtual de Python.
* **El módulo `dis` (Disassembler):** Aprender a leer el "ensamblador" de Python para entender qué hace el intérprete bajo el capó cuando sumas dos variables o usas un bucle `for`.


* **Práctica:**


In [2]:
import dis
def suma(a, b):
    return a + b
dis.dis(suma) # Verás instrucciones como LOAD_FAST, BINARY_OP, RETURN_VALUE


  2           RESUME                   0

  3           LOAD_FAST_LOAD_FAST      1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE



```



---

### Módulo 4: Gestión de Memoria y el Recolector de Basura (Garbage Collector)

*Cómo Python administra los recursos de tu computador.*

* **Conteo de referencias (Reference Counting):** La estrategia principal de Python. Cada objeto tiene un contador que rastrea cuántas variables lo apuntan. Si llega a cero, el objeto se destruye inmediatamente.
* **El Recolector de Basura Generacional (`gc`):** ¿Qué pasa con las **referencias cíclicas** (el objeto A apunta al B, y el B apunta al A)? Aquí es donde entra el recolector de basura analizando grafos dirigidos para detectar componentes fuertemente conexos aislados y liberar memoria.
* **Análogo Matemático:** Teoría de grafos dirigida. Identificar ciclos en digrafos donde los nodos son variables en memoria.

---



### Módulo 5: Concurrencia y el fin del GIL (El Santo Grial de Python 3.13+)

*El tema más caliente de la PyCon.*

* **¿Qué es el GIL (Global Interpreter Lock)?** Históricamente, Python no podía ejecutar múltiples hilos (*threads*) en paralelo en múltiples núcleos de CPU porque el GIL protegía la memoria de CPython de colisionar.
* **El cambio histórico (PEP 703):** Daniel hablará de "Python sin GIL" (Free-threaded Python). Tienes que entender los conceptos de:
* **Concurrencia vs. Paralelismo.**
* **Race Conditions** (Condiciones de carrera) y cómo el equipo de desarrollo de Python eliminó el GIL sin destruir el rendimiento de los programas existentes.


* **¿Qué es un PEP?** (*Python Enhancement Proposal*). Entender el proceso democrático y técnico mediante el cual la comunidad propone y decide los cambios matemáticos y lógicos en el lenguaje.



---

## Recursos recomendados para empezar hoy

Dado que vienes de VS Code y Jupyter, te sugiero hacer a un lado temporalmente a Pandas/Numpy para estos experimentos y enfocarte en el Python puro (Vanilla Python).

1. **Libro Clave:** *CPython Internals* de Anthony Shaw. Es la guía perfecta para entender cómo está programado Python por dentro sin necesidad de ser un experto en lenguaje C.
2. **Herramienta Visual:** [Python Tutor](https://pythontutor.com/). Te permite escribir código y ver gráficamente cómo se crean los objetos en memoria y cómo funcionan las referencias en tiempo real.
3. **Documentación Oficial:** Lee sobre el módulo `sys` (especialmente `sys.getrefcount`) y el módulo `gc`.

# Empecemos con el estudio del módulo 1 

El Módulo 1 es el cimiento de todo. Al final de esta sesión, verás tu entorno de Jupyter y VS Code con una perspectiva completamente diferente.



Como matemático, estás muy familiarizado con la distinción entre un **objeto abstracto** (definido por axiomas y reglas) y sus **representaciones o realizaciones concretas**. En la computación ocurre exactamente lo mismo, y esa es la primera gran barrera mental que vamos a derribar.

---



## 1. La gran distinción: Especificación vs. Implementación

Cuando la gente dice "Python", suele mezclar dos conceptos que son técnicamente independientes:

1. **La Especificación de Python (El "Qué"):** Es un documento de texto (el *Python Language Reference*). Define las reglas formales, la sintaxis y la semántica del lenguaje. Por ejemplo: *"La palabra clave `def` inicia la definición de una función"*. Esto es puramente abstracto. Es el equivalente a los **axiomas y teoremas** de un sistema matemático.
2. **La Implementación de Python (El "Cómo"):** Es un programa real que alguien escribió para tomar un archivo de texto con código Python, entenderlo y ejecutarlo en un procesador físico. Esto es equivalente a un **modelo** concreto que satisface esos axiomas.



Como la especificación es libre, cualquiera puede escribir su propio "traductor" de Python. Por eso existen múltiples intérpretes:

* **CPython:** Es la implementación de referencia, escrita en **lenguaje C**. Es el Python estándar que descargaste con Anaconda o desde python.org. Es el motor que corre detrás de tu VS Code y tus cuadernos de Jupyter.
* **PyPy:** Un intérprete escrito en el propio Python (una versión restringida llamada RPython) enfocado en la velocidad. Utiliza compilación *JIT (Just-In-Time)* para traducir código a lenguaje máquina en caliente.
* **Jython / IronPython:** Intérpretes escritos en Java y C# (.NET) respectivamente, que permiten a Python interactuar directamente con esos ecosistemas.

---



## 2. Anatomía de CPython: ¿Qué pasa cuando ejecutas una celda en Jupyter?

Cuando estás en Jupyter y presionas `Shift + Enter` en una celda, o cuando ejecutas un archivo `.py` en VS Code, se desencadena una línea de ensamblaje muy rigurosa dentro de **CPython**.



A diferencia de lenguajes como C o Rust (que se compilan directamente a código de máquina nativo del procesador), o lenguajes puramente interpretados línea por línea (como las calculadoras antiguas), Python utiliza un **enfoque híbrido**.



Este es el viaje que realiza tu código:

```
                  [ Tu código en texto plano ] 
                               │
                               ▼
 1. Compilación ──────▶ [ El Analizador (Parser) ]
                         Genera el Árbol de Sintaxis (AST)
                               │
                               ▼
                        [ El Compilador ]
                         Genera el Bytecode (.pyc)
                               │
                               ▼
 2. Ejecución ────────▶ [ La Máquina Virtual (PVM) ]
                         Ejecuta las instrucciones en la CPU

```



### Paso 1: El Analizador (Parser) y el AST

El intérprete no entiende tu texto tal cual. Lo primero que hace es romper el texto en "tokens" (palabras clave, números, operadores) y verificar si la sintaxis es matemáticamente válida usando una gramática formal. Si es válida, construye una estructura de datos jerárquica llamada **AST (Árbol de Sintaxis Abstracta)**.
*Si hay un error aquí, tu código ni siquiera empieza a ejecutarse; se lanza un `SyntaxError`.*



### Paso 2: El Compilador de CPython

El compilador toma ese AST (el árbol) y lo traduce a una serie de instrucciones de bajo nivel llamadas **Bytecode**. Este bytecode se guarda en memoria (y a veces en el disco duro dentro de esas carpetas misteriosas llamadas `__pycache__` con extensión `.pyc`).
El bytecode es un lenguaje intermedio; no es código que entienda tu procesador Intel, AMD o Apple Silicon, sino que está diseñado para una CPU hipotética e idealizada.



### Paso 3: La Máquina Virtual de Python (PVM)

Aquí es donde ocurre la magia de la ejecución. La PVM (Python Virtual Machine) es un gran bucle en C que toma cada instrucción de bytecode, la interpreta y la ejecuta usando los recursos de tu computador real.

---



## 3. Práctica: Mirando debajo de la alfombra en tu Jupyter o VS Code

Vamos a hacer un experimento interactivo rápido. Abre una celda de tu Jupyter Notebook o un archivo en VS Code y ejecuta el siguiente código:


In [7]:
import sys

# Definimos una variable simple
mi_numero = 2**10  # Un número entero muy grande

# Preguntémosle a CPython (que está escrito en C) 
# cuántos bytes de memoria física está usando para almacenar este entero.
tamano_en_bytes = sys.getsizeof(mi_numero)

print(f"El número {mi_numero} está usando {tamano_en_bytes} bytes en memoria.")


El número 1024 está usando 28 bytes en memoria.



## El misterio matemático para analizar:

Si ejecutas esto, verás que el número `1024` ocupa aproximadamente **28 bytes** (el número exacto puede variar ligeramente según tu sistema operativo).



Desde el punto de vista de la matemática aplicada o la computación clásica, el número entero `1024` se puede representar en binario como $2^{10}$, lo cual requiere apenas **2 bytes** (16 bits) de información.

**¿Por qué Python necesita 28 bytes para guardar un simple entero?**
Porque en CPython, **todo es un objeto de C** bajo el capó (específicamente una estructura llamada `PyObject`). Ese número de 28 bytes no solo guarda el valor `1024`, sino que también guarda:

1. Un contador de referencias (para saber cuándo borrarlo de la memoria).
2. Un puntero al tipo de objeto (que le dice a Python: "Oye, soy un entero (`int`)").
3. El tamaño del número (ya que Python admite enteros de precisión arbitraria; es decir, ¡puedes calcular $2^{10000}$ sin que ocurra un desbordamiento de memoria!).

---


In [8]:
2**10_000

1995063116880758384883742162683585083823496831886192454852008949852943883022194663191996168403619459789933112942320912427155649134941378111759378593209632395785573004679379452676524655126605989552055008691819331154250860846061810468550907486608962488809048989483800925394163325785062156830947390255691238806522509664387444104675987162698545322286853816169431577562964076283688076073222853509164147618395638145896946389941084096053626782106462142733339403652556564953060314268023496940033593431665145929777327966577560617258203140799419817960737824568376228003730288548725190083446458145465055792960141483392161573458813925709537976911927780082695773567444412306201875783632550272832378927071037380286639303142813324140162419567169057406141965434232463880124885614730520743199225961179625013099286024170834080760593232016126849228849625584131284406153673895148711425631511108974551420331382020293164095759646475601040584584156607204496286701651506192063100418642227590867090057460641785695191145605506


### Tu turno de experimentar:

1. Corre el código anterior en tu entorno actual.
2. Prueba cambiando `mi_numero` por un número gigantesco, por ejemplo `2**100` (dos elevado a la cien), y vuelve a medir su tamaño con `sys.getsizeof`. ¿Cuánto aumentó?
3. Cuéntame qué valor te dio y dime: con lo que hemos visto hoy, **¿cómo definirías ahora qué es "el intérprete de Python" en tus propias palabras?**

El interprete de Python es un programa de un lenguaje de programación como C que lee el código Python y lo convierte en lenguaje máquina ejecutable por mi CPU

¡Excelente respuesta! Tu capacidad para destilar conceptos complejos en definiciones directas se nota de inmediato.

Sin embargo, dado que nos estamos preparando para una charla de nivel **Avanzado** en la PyCon, vamos a hacer una **precisión técnica muy importante** (un pequeño ajuste de tuerca) que es justamente el corazón de lo que Daniel Arango explicará en su conferencia.

Aquí está la evaluación de tu respuesta:

### Lo que está **perfecto**:

* **"Es un programa escrito en otro lenguaje (como C)..."**
¡Exacto! Identificaste con precisión que el intérprete estándar (CPython) es un software escrito en C. No es magia, es solo código de C interactuando con tu sistema operativo.
* **"... que lee el código Python..."**
Correcto, toma el texto plano que tú escribes en VS Code o Jupyter y lo procesa.

---

### El detalle técnico a corregir (¡El "Aha!" de la charla!):

* En tu respuesta dices: **"... y lo convierte en lenguaje máquina ejecutable por mi CPU"**.

**Aquí está el sutil pero crucial error:** El intérprete estándar de Python (CPython) **no** traduce tu código a lenguaje máquina (el código binario nativo de tu procesador Intel, AMD o Apple Silicon).

Quien hace eso es un **compilador puro** (como el de C, C++ o Rust).

Lo que hace el intérprete de Python en realidad es:

1. Traducir tu código a **Bytecode** (un lenguaje intermedio, una abstracción).
2. La **Máquina Virtual de Python (PVM)** lee ese bytecode instrucción por instrucción y ella misma realiza las operaciones en C. Tu CPU real ejecuta el código de C del intérprete, no tu código de Python directamente.

> **La analogía matemática:**
> Imagina que tienes un libro de geometría escrito en **Alemán** (Código Python). Tu CPU física solo habla **Chino** (Lenguaje Máquina).
> Un compilador tradicional tomaría el libro en Alemán y escribiría un libro completamente nuevo en Chino.
> El intérprete de Python (CPython), en cambio, es un **traductor humano** que sabe Alemán y Chino. El traductor lee el libro en Alemán, lo procesa en su "mente" usando un lenguaje intermedio (Bytecode) y él mismo realiza las acciones o te lo explica en Chino sobre la marcha. La CPU solo interactúa con el traductor (el programa en C), nunca recibe un archivo en Chino.

---

### Tu definición optimizada para la PyCon:

Si tuvieras que explicarlo en el congreso, tu definición refinada sería:

> *"El intérprete de Python es un programa (generalmente escrito en C) que toma nuestro código en texto plano, lo traduce a un formato intermedio abstracto llamado **bytecode**, y luego ejecuta ese bytecode paso a paso utilizando su propia **máquina virtual**, actuando como un intermediario entre nuestro código y la CPU física."*

¿Hace sentido esta distinción entre "lenguaje máquina" y "bytecode/máquina virtual"? Si es así, ¡estamos listos para dar el salto al **Módulo 2: Gramáticas, Parsers y AST**, donde veremos cómo se construye ese "árbol" antes de generar el bytecode!

El Módulo 2 es un territorio fascinante para un matemático. En la carrera de matemáticas se estudian estructuras algebraicas y teoría de grafos de manera puramente abstracta. Aquí verás cómo la ciencia de la computación utiliza esas mismas teorías para lograr que una máquina "entienda" la lógica humana.

---



## Módulo 2: Gramáticas, Analizadores (Parsers) y el Árbol de Sintaxis Abstracta (AST)

Antes de que Python genere el bytecode, debe estructurar tu código. Esto lo hace mediante dos pasos fundamentales: **parsing** (análisis sintáctico) y la construcción del **AST**.



### 1. El Parser: ¿Cómo se lee una Gramática Formal?

Para una computadora, tu archivo de Python es solo una larga cadena de caracteres (un string gigante). El **Parser** es el algoritmo encargado de agrupar esos caracteres en estructuras sintácticas válidas utilizando reglas matemáticas llamadas **Gramáticas Libres de Contexto**.



Daniel Arango mencionará en su charla la transición histórica que vivió Python entre dos tipos de parsers:

* **El viejo Parser LL(1):** Analizaba el código de izquierda a derecha (*Left-to-right*), construyendo una derivación por la izquierda (*Leftmost derivation*) utilizando solo **1 token de anticipación** (*lookahead*). El problema es que las gramáticas LL(1) son muy restrictivas matemáticamente y no permiten recursión por la izquierda (por ejemplo, definir una regla del tipo $A \to A + B$ causaba un bucle infinito en el parser).
* **El nuevo Parser PEG (Parsing Expression Grammars):** Introducido en Python 3.9 (PEP 617). Elimina la ambigüedad porque las elecciones son ordenadas (si la primera regla funciona, se queda con esa). Permite un análisis mucho más intuitivo, recursión por la izquierda y un rendimiento excelente sin las limitaciones matemáticas de LL(1).



### 2. ¿Qué es un AST?

Una vez que el parser valida la sintaxis, el compilador construye el **Árbol de Sintaxis Abstracta (AST)**.

> En términos matemáticos, un AST es un **grafo dirigido, acíclico y conexo (un árbol enraizado y ordenado)**. Los nodos internos representan operadores u operaciones lógicas (como sumas, asignaciones de variables o bucles), y las hojas representan operandos (valores literales como números o variables).
<img src = 'https://ruslanspivak.com/lsbasi-part7/lsbasi_part7_ast_01.png'>
Como puedes ver en la imagen, mientras que el *Parse Tree* (árbol de derivación sintáctica) contiene todos los detalles gramaticales y nodos redundantes del lenguaje, el **AST** (a la izquierda) es una representación simplificada y optimizada que preserva únicamente las operaciones matemáticas reales: multiplicar $2 \times 7$ y luego sumarle $3$.

---



## Ejemplos de Aplicación Práctica con el módulo `ast` de Python

Para entender de verdad qué puedes hacer con el AST (lo cual es el núcleo de las herramientas de análisis de código moderno), veamos tres ejemplos prácticos que puedes ejecutar en tu Jupyter Notebook o VS Code.



### Ejemplo 1: El analizador estático (Linter casero)

Imagina que eres profesor de programación y quieres escribir un script en Python que analice automáticamente las tareas de tus alumnos para **contar cuántos bucles `for` utilizaron**, sin tener que abrir los archivos ni buscarlos usando expresiones regulares (que fallarían si hay comentarios).



In [10]:
import ast

# Este es el código del estudiante que vamos a analizar (como un string)
codigo_alumno = """
def calcular_potencias(limite):
    resultados = []
    for i in range(limite):
        for j in range(2):
            resultados.append(i ** j)
    return resultados
"""

# Convertimos el string de código en un Árbol de Sintaxis Abstracta (AST) real
arbol = ast.parse(codigo_alumno)

# Creamos un "Visitador" de nodos. 
# Esta es una clase que recorre el grafo del árbol buscando tipos de nodos específicos.
class ContadorDeBucles(ast.NodeVisitor):
    def __init__(self):
        self.contador_for = 0

    # Este método se activa automáticamente cada vez que el visitante encuentra un nodo tipo 'For'
    def visit_For(self, node):
        self.contador_for += 1
        self.generic_visit(node) # Sigue recorriendo los hijos del nodo

# Ejecutamos nuestro analizador
analizador = ContadorDeBucles()
analizador.visit(arbol)

print(f"El alumno usó {analizador.contador_for} bucles 'for' en su código.")


El alumno usó 2 bucles 'for' en su código.



## Ejemplo 2: Modificación dinámica de código (Mutación del AST)



Este es un truco mental avanzado: **podemos alterar el álgebra del código antes de que se ejecute**.
Vamos a interceptar el AST y, mediante un transformador de nodos, cambiaremos mágicamente todas las operaciones de suma (`+`) por restas (`-`).


In [11]:
import ast

codigo_original = "resultado = 10 + 5 + 3"

# Parseamos el código
arbol = ast.parse(codigo_original)

# Creamos un Transformador de Nodos del AST
class MutadorAlgebraico(ast.NodeTransformer):
    # Cuando encuentre una suma (Add), la transformará en una resta (Sub)
    def visit_Add(self, node):
        return ast.Sub()

# Aplicamos la transformación al árbol
arbol_mutado = MutadorAlgebraico().visit(arbol)

# Fijamos las líneas del nuevo árbol para que sea ejecutable
ast.fix_missing_locations(arbol_mutado)

# Compilamos el árbol mutado a bytecode y lo ejecutamos
bytecode_mutado = compile(arbol_mutado, filename="<string>", mode="exec")
variables_locales = {}
exec(bytecode_mutado, {}, variables_locales)

print("Código original: resultado = 10 + 5 + 3")
print("Resultado real de la ejecución mutada:", variables_locales["resultado"])
# El resultado será 2 (es decir, calculó 10 - 5 - 3 bajo el capó)


Código original: resultado = 10 + 5 + 3
Resultado real de la ejecución mutada: 2



*Este tipo de técnicas se utilizan en herramientas de "Mutation Testing" para probar si las pruebas unitarias de un software de verdad detectan cambios maliciosos o errores lógicos en el código fuente.*



### Ejemplo 3: Evaluación segura de expresiones matemáticas (`eval` vs `literal_eval`)

Seguramente has escuchado que usar la función nativa `eval()` en Python es extremadamente peligroso, porque si evalúas un string enviado por un usuario, este podría contener comandos para borrar tu disco duro.
La librería `ast` proporciona una alternativa matemáticamente segura que analiza el string, construye el AST y solo permite compilarlo si se compone exclusivamente de tipos de datos literales (números, strings, tuplas, listas).



In [9]:
import ast

expresion_segura = "[1, 2, 3, (4, 5)]"
expresion_peligrosa = "__import__('os').system('echo HACKEADO')" # Intento de ejecutar comandos del sistema

# Uso de literal_eval
try:
    resultado = ast.literal_eval(expresion_segura)
    print("Evaluación exitosa (Estructura segura):", resultado)
except ValueError:
    print("No es una estructura válida de Python")

try:
    # Esto lanzará un error de valor en lugar de ejecutar el comando destructivo
    resultado_peligroso = ast.literal_eval(expresion_peligrosa)
except ValueError:
    print("¡Bloqueado! Se detectó código ejecutable no permitido en el string.")


Evaluación exitosa (Estructura segura): [1, 2, 3, (4, 5)]
¡Bloqueado! Se detectó código ejecutable no permitido en el string.


## Tu turno de analizar:

Mira con atención el **Ejemplo 2** (donde cambiamos las sumas por restas). Daniel Arango menciona en su biografía: *"compartiré historias personales, incluyendo batallas causadas por ASTs idénticos"*.

Basándote en lo que acabas de ver sobre cómo funciona un AST:

1. ¿Cómo crees que dos fragmentos de código escritos de forma diferente podrían producir el mismo AST?
2. O al revés, ¿cómo una pequeña modificación en el orden sintáctico podría alterar drásticamente el árbol y causar una "batalla" en tiempo de ejecución?

# Exploremos las dos preguntas anteriores  

Como matemático, este es uno de los temas más hermosos porque tiene que ver directamente con la **equivalencia lógica** y la **semántica de los lenguajes formales**.

Vamos a resolver ambos misterios: cómo dos códigos distintos generan el mismo AST (un problema de "muchos a uno") y cómo un cambio minúsculo desmorona el árbol por completo.

---



## Misterio 1: ¿Cómo dos códigos diferentes generan un AST idéntico?

En matemáticas, sabemos que la expresión $x + y$ es equivalente a $y + x$ debido a la propiedad conmutativa, pero sintácticamente están escritas de forma distinta.

En la programación, a veces escribimos código con "ruido" visual o diferencias de estilo que no cambian en absoluto la lógica de ejecución. El AST es **abstracto** precisamente porque se deshace de ese ruido.



### Ejemplo 1: Paréntesis redundantes

En Python, puedes escribir esto:



In [ ]:
# Opción A
x = 5 + 3

# Opción B
x = (5 + 3)



Para el parser de Python, los paréntesis en la Opción B solo sirven para agrupar, pero una vez que se determina la prioridad de la operación, **desaparecen**. Ambas opciones generan exactamente el mismo AST:

```
Assign
├── targets: [Name(id='x')]
└── value: BinOp
    ├── left: Constant(value=5)
    ├── op: Add
    └── right: Constant(value=3)

```

No hay ningún nodo "Parenthesis". El AST para ambos es idéntico.



### Ejemplo 2: El espaciado y los saltos de línea (Indentación vs. Formato)

Python es famoso por usar la indentación para definir bloques, pero el uso de espacios en blanco *dentro* de una línea no afecta la lógica:



In [12]:
# Opción A
resultado=10+2
# Opción B
resultado   =   10   +   2



Ambos strings de código producen un AST idéntico porque las distancias físicas entre caracteres son ignoradas por el analizador léxico (*lexer*) al generar los tokens.

> **¿Dónde ocurre la "batalla" de la que habla Daniel?**
> Imagina que estás construyendo una herramienta que detecta plagio en tareas de programación comparando los ASTs de los estudiantes. Si un estudiante copió el código de otro y solo le agregó paréntesis redundantes o cambió los espacios, sus ASTs serán **100% idénticos**. ¡El plagio queda al descubierto de inmediato!

---



## Misterio 2: ¿Cómo un cambio minúsculo altera drásticamente el árbol?

Aquí es donde la precedencia de operadores (el orden de las operaciones en aritmética) y la estructura del árbol chocan fuertemente. Un cambio de un solo caracter puede reestructurar el grafo por completo, cambiando la "altura" de los nodos o la dirección de las ramas.

### Caso: Una coma perdida transforma un entero en una tupla

Este es uno de los errores más comunes y frustrantes en Python. Mira estos dos códigos:


In [ ]:
# Código 1
mi_variable = (42)

# Código 2
mi_variable = (42,)



Sintácticamente, la única diferencia es una coma (`,`). Pero en el AST, el cambio es radical:

* **En el Código 1:** El AST ve un nodo `Assign` cuyo valor es simplemente una constante entera `Constant(value=42)`. Los paréntesis son ignorados.
* **En el Código 2:** La presencia de la coma le dice al parser: *"¡Esto es una tupla!"*. El AST ahora tiene un nodo `Tuple` que contiene dentro un nodo `Constant(value=42)`.

Si tu código más adelante esperaba un número para hacer una operación matemática ($42 + 8$), el Código 2 romperá el programa con un `TypeError` porque no puedes sumarle un número a una tupla.



### Caso: Precedencia de Operadores y la forma del árbol

Consideremos estas dos expresiones aritméticas:

```
Expresión A: 2 + 3 * 5
Expresión B: (2 + 3) * 5

```



En la **Expresión A**, debido a que la multiplicación tiene mayor prioridad que la suma, el árbol se construye colocando la multiplicación en la parte inferior (se evalúa primero) y la suma en la raíz:

```
      +  (Suma es la raíz)
     / \
    2   *
       / \
      3   5

```

En la **Expresión B**, los paréntesis fuerzan a que la suma sea lo primero en evaluarse, invirtiendo la estructura del árbol por completo:

```
        *  (Multiplicación es la raíz)
       / \
      +   5
     / \
    2   3

```



Como ves, la estructura geométrica del grafo cambia drásticamente. En compiladores, la "forma" de este árbol determina el orden en que la CPU ejecutará las instrucciones.

---



Con esto hemos cerrado el **Módulo 2**. Tienes una comprensión sólida de cómo Python procesa el texto, lo convierte en un árbol lógico y cómo ese árbol representa fielmente la semántica de tu código.



¿Cómo te sientes para avanzar al **Módulo 3: El Bytecode y el Desensamblador**, donde veremos cómo ese árbol se aplana para convertirse en instrucciones de bajo nivel?

# Continuemos con el módulo 3  

El Módulo 3 es donde la abstracción matemática del árbol (AST) se "aplana" y se convierte en una secuencia lineal de instrucciones de bajo nivel.

Como matemático, estás acostumbrado a la idea de **proyecciones** u **homomorfismos**: tomar una estructura rica y compleja en un espacio dimensional superior (el árbol AST) y proyectarla a un espacio unidimensional conservando sus propiedades esenciales. Eso es exactamente lo que hace el compilador de Python al generar el **Bytecode**.

---



## Módulo 3: El Bytecode y el Desensamblador

A diferencia de un procesador real (como tu chip Intel o AMD), que tiene "registros" físicos de hardware para almacenar datos temporalmente durante un cálculo, la Máquina Virtual de Python (PVM) es una **máquina basada en pila (Stack-based Virtual Machine)**.



Esto significa que todas las operaciones ocurren en una estructura de datos tipo LIFO (Last In, First Out) llamada **pila de evaluación**.



### Las instrucciones básicas del Bytecode

Casi todo el bytecode de Python se reduce a tres tipos de acciones en la pila:

1. `LOAD_CONST` o `LOAD_FAST`: Pone un valor o variable en la parte superior de la pila.
2. Operaciones matemáticas/lógicas (como `BINARY_OP` o `COMPARE_OP`): Saca los dos valores superiores de la pila, realiza la operación y vuelve a meter el resultado en la pila.
3. `STORE_FAST` o `STORE_NAME`: Saca el valor superior de la pila y lo guarda en una variable.

---



## 🛠️ Práctica: Desensamblando código en tu Jupyter o VS Code

Para ver este proceso en vivo, usaremos el módulo `dis` de la librería estándar de Python. Este módulo es el "desensamblador" oficial de Python.

### Ejemplo 1: El cálculo de una hipotenusa

Vamos a escribir una función simple que calcule el cuadrado de la hipotenusa: $h^2 = a^2 + b^2$. Ejecuta esto en tu celda de Jupyter:

```python


In [13]:
import dis

def calcular_hipotenusa_cuadrada(a, b):
    return a**2 + b**2

# Le pedimos a Python que nos muestre el bytecode de la función
dis.dis(calcular_hipotenusa_cuadrada)


  3           RESUME                   0

  4           LOAD_FAST                0 (a)
              LOAD_CONST               1 (2)
              BINARY_OP                8 (**)
              LOAD_FAST                1 (b)
              LOAD_CONST               1 (2)
              BINARY_OP                8 (**)
              BINARY_OP                0 (+)
              RETURN_VALUE



Al ejecutarlo, verás una salida en consola similar a esta:

```text
  2           0 RESUME                   0

  3           2 LOAD_FAST                0 (a)
              4 LOAD_CONST               1 (2)
              6 BINARY_OP                5 (**)
             10 LOAD_FAST                1 (b)
             12 LOAD_CONST               1 (2)
             14 BINARY_OP                5 (**)
             18 BINARY_OP                0 (+)
             22 RETURN_VALUE

```



#### Anatomía de la salida (El análisis paso a paso):

* **La columna de la izquierda (`2`, `3`):** Indica la línea de código de tu archivo original.
* **Los números (`0`, `2`, `4`...):** Son las direcciones de memoria (en bytes) de cada instrucción.
* **La columna central (`LOAD_FAST`, `BINARY_OP`):** Es el código de operación (*opcode*).
* **La columna derecha:** Los argumentos que recibe esa instrucción.

**El viaje de la pila para este cálculo:**

1. `LOAD_FAST 0 (a)`: Mete el valor de la variable `a` en la pila. `[a]`
2. `LOAD_CONST 1 (2)`: Mete el número `2` en la pila. `[a, 2]`
3. `BINARY_OP (**)`: Saca `a` y `2`, calcula $a^2$ y mete el resultado. `[a**2]`
4. `LOAD_FAST 1 (b)`: Mete el valor de `b` en la pila. `[a**2, b]`
5. `LOAD_CONST 1 (2)`: Mete el número `2`. `[a**2, b, 2]`
6. `BINARY_OP (**)`: Saca `b` y `2`, calcula $b^2$ y lo mete. `[a**2, b**2]`
7. `BINARY_OP (+)`: Saca los dos resultados anteriores, los suma y mete el total. `[a**2 + b**2]`
8. `RETURN_VALUE`: Saca el resultado final y lo devuelve a quien llamó la función.

---



### Ejemplo 2: El costo oculto de las variables (Optimización matemática)

Como matemático, te gustará ver cómo un pequeño cambio de estilo afecta la eficiencia del cómputo. Comparemos estas dos funciones equivalentes que calculan $f(x) = x^2 + x^2$:



In [14]:
import dis

def f1(x):
    # Guardamos el término en una variable intermedia
    termino = x**2
    return termino + termino

def f2(x):
    # Hacemos el cálculo directamente
    return x**2 + x**2

print("--- BYTECODE DE f1 ---")
dis.dis(f1)
print("\n--- BYTECODE DE f2 ---")
dis.dis(f2)


--- BYTECODE DE f1 ---
  3           RESUME                   0

  5           LOAD_FAST                0 (x)
              LOAD_CONST               1 (2)
              BINARY_OP                8 (**)
              STORE_FAST               1 (termino)

  6           LOAD_FAST_LOAD_FAST     17 (termino, termino)
              BINARY_OP                0 (+)
              RETURN_VALUE

--- BYTECODE DE f2 ---
  8           RESUME                   0

 10           LOAD_FAST                0 (x)
              LOAD_CONST               1 (2)
              BINARY_OP                8 (**)
              LOAD_FAST                0 (x)
              LOAD_CONST               1 (2)
              BINARY_OP                8 (**)
              BINARY_OP                0 (+)
              RETURN_VALUE



Si comparas las salidas, notarás que `f1` tiene instrucciones adicionales como `STORE_FAST` (para guardar la variable `termino`) y luego múltiples `LOAD_FAST` para volver a leerla.

En Python, acceder y asignar variables locales tiene un costo en ciclos de CPU. Por lo tanto, `f2` no solo es más directa, sino que genera menos instrucciones en la máquina virtual, haciéndola marginalmente más rápida.

---



### Ejemplo 3: ¿Por qué `f"{x}"` es más rápido que `"x = " + str(x)`?

Una duda muy común es cuál es la forma más eficiente de concatenar texto. El desensamblador nos da la respuesta inmediata de por qué los f-strings de Python son tan veloces:


In [15]:

import dis

def concatenar_viejo(x):
    return "Numero: " + str(x)

def concatenar_fstring(x):
    return f"Numero: {x}"

print("--- CONCATENAR VIEJO ---")
dis.dis(concatenar_viejo)
print("\n--- F-STRING ---")
dis.dis(concatenar_fstring)



--- CONCATENAR VIEJO ---
  3           RESUME                   0

  4           LOAD_CONST               1 ('Numero: ')
              LOAD_GLOBAL              1 (str + NULL)
              LOAD_FAST                0 (x)
              CALL                     1
              BINARY_OP                0 (+)
              RETURN_VALUE

--- F-STRING ---
  6           RESUME                   0

  7           LOAD_CONST               1 ('Numero: ')
              LOAD_FAST                0 (x)
              FORMAT_SIMPLE
              BUILD_STRING             2
              RETURN_VALUE



En la versión vieja, verás que Python tiene que buscar la función global `str` en memoria (`LOAD_GLOBAL`), llamarla (`CALL`), y luego realizar una suma de strings (`BINARY_OP`).

En cambio, con el f-string, Python utiliza una instrucción especializada de bajo nivel llamada `FORMAT_VALUE` diseñada directamente en C para optimizar la conversión a texto en un solo paso, ¡esquivando la costosa búsqueda de la función global `str`!

---



### Tu turno de experimentar:

Escribe una función muy sencilla en tu Jupyter Notebook, desensámblala usando `dis.dis()` y observa el resultado.

Por ejemplo, prueba con una operación lógica de control:


In [21]:

def es_par1(n):
    if n % 2 == 0:
        return True
    return False


def es_par2(n):
    return n % 2 == 0


dis.dis(es_par1)

dis.dis(es_par2) 

  1           RESUME                   0

  2           LOAD_FAST                0 (n)
              LOAD_CONST               1 (2)
              BINARY_OP                6 (%)
              LOAD_CONST               2 (0)
              COMPARE_OP              88 (bool(==))
              POP_JUMP_IF_FALSE        1 (to L1)

  3           RETURN_CONST             3 (True)

  4   L1:     RETURN_CONST             4 (False)
  7           RESUME                   0

  8           LOAD_FAST                0 (n)
              LOAD_CONST               1 (2)
              BINARY_OP                6 (%)
              LOAD_CONST               2 (0)
              COMPARE_OP              72 (==)
              RETURN_VALUE


In [20]:
dis.dis(es_par1)

dis.dis(es_par2) 

  1           RESUME                   0

  2           LOAD_FAST                0 (n)
              LOAD_CONST               1 (2)
              BINARY_OP                6 (%)
              LOAD_CONST               2 (0)
              COMPARE_OP              88 (bool(==))
              POP_JUMP_IF_FALSE        1 (to L1)

  3           RETURN_CONST             3 (True)

  4   L1:     RETURN_CONST             4 (False)
  7           RESUME                   0

  8           LOAD_FAST                0 (n)
              LOAD_CONST               1 (2)
              BINARY_OP                6 (%)
              LOAD_CONST               2 (0)
              COMPARE_OP              72 (==)
              RETURN_VALUE



Si la desensamblas, verás aparecer instrucciones de salto como `POP_JUMP_IF_FALSE` (que equivale al "ir a" o salto condicional en lógica de bajo nivel).

¿Pudiste ver la pila de operaciones en acción? ¿Tiene sentido cómo Python linealiza los cálculos matemáticos para su máquina virtual?

Ahora entiendo que linealizar es construit mejor un código equivalente pero sin bifurcaciones. 

Este es un ejercicio analítico fantástico. Como matemático, tu intuición de seguro te dice que `es_par1` y `es_par2` son lógicamente equivalentes: ambas son representaciones de la función indicadora del conjunto de los números pares.

Sin embargo, a la luz del **bytecode y la máquina virtual de Python**, sus proyecciones lineales son muy diferentes en términos de eficiencia. `es_par2` es considerablemente más elegante y rápida para la CPU.

Desensamblemos ambas funciones mentalmente y analicemos sus bytecodes estructurales para ver por qué.

---

## Análisis de `es_par1(n)` (El enfoque imperativo con control de flujo)

Tu primera función tiene un bloque condicional explícito (`if/else` implícito). Su bytecode se ve estructurado de la siguiente forma en CPython:

```text
  2           0 LOAD_FAST                0 (n)
              2 LOAD_CONST               1 (2)
              4 BINARY_OP                6 (%)
              8 LOAD_CONST               2 (0)
             10 COMPARE_OP               2 (==)
             14 POP_JUMP_IF_FALSE        20  <-- [El Salto Condicional]

  3          16 LOAD_CONST               3 (True)
             18 RETURN_VALUE

  4     >>   20 LOAD_CONST               4 (False)
             22 RETURN_VALUE

```

### ¿Qué está ocurriendo aquí?

1. **Línea 2:** Python calcula el residuo (`n % 2`), luego mete el `0` a la pila y ejecuta `COMPARE_OP (==)`. El resultado de esa comparación (`True` o `False`) queda en la cima de la pila.
2. **El cuello de botella (`POP_JUMP_IF_FALSE`):** Aquí la Máquina Virtual tiene que tomar una decisión. Lee el valor de la pila. Si el número **no** era par (es decir, la comparación dio `False`), la PVM tiene que alterar su contador de programa y "saltar" directamente a la dirección `20` (la línea 4 del código), saltándose los pasos 16 y 18.
3. **El retorno:** Dependiendo de si saltó o no, carga `True` o `False` y ejecuta `RETURN_VALUE`.

---

## Análisis de `es_par2(n)` (El enfoque funcional/algebraico)

Tu segunda función elimina el condicional por completo y devuelve directamente el resultado de la proposición lógica. Su bytecode es asombrosamente limpio y compacto:

```text
  7           0 LOAD_FAST                0 (n)
              2 LOAD_CONST               1 (2)
              4 BINARY_OP                6 (%)
              8 LOAD_CONST               2 (0)
             10 COMPARE_OP               2 (==)
             14 RETURN_VALUE

```

### ¿Qué está ocurriendo aquí?

1. Carga `n`, carga `2`, calcula el residuo (`%`).
2. Carga `0`.
3. Ejecuta `COMPARE_OP (==)`. En este punto exacto, el resultado de evaluar la proposición lógica ya es un objeto booleano (`True` o `False`) que reside en la cima de la pila.
4. **Final inmediato:** En lugar de evaluar qué valor tiene ese booleano para decidir a dónde saltar, Python simplemente ejecuta `RETURN_VALUE` con lo que sea que esté en la pila.

---

## 📊 Comparación y Conclusión para la PyCon

Al poner una función al lado de la otra, podemos extraer tres conclusiones clave a nivel de arquitectura de software:

| Métrica | `es_par1(n)` | `es_par2(n)` |
| --- | --- | --- |
| **Número de instrucciones** | 9-10 instrucciones | 6 instrucciones |
| **Complejidad del Grafo** | Bifurcado (Tiene dos caminos posibles) | Lineal (Un solo camino directo) |
| **Instrucciones de Salto** | Sí (`POP_JUMP_IF_FALSE`) | No (Ninguna) |



### ¿Por qué `es_par2` es superior?

1. **Menos Bytecode:** Requiere casi un 40% menos de instrucciones dentro de la Máquina Virtual.
2. **Sin penalización por predicción de saltos:** En CPUs modernas (e incluso a nivel de la máquina virtual de CPython), los saltos condicionales obligan al motor a evaluar condiciones de flujo. `es_par2` fluye en línea recta; no hay decisiones que tomar, solo operaciones que calcular.
3. **Reducción de ruido en el AST:** El árbol sintáctico de `es_par1` tiene nodos de control de flujo (`If`), mientras que el de `es_par2` es un árbol puramente de retorno de expresión (`Return -> Compare`).

Como matemático, este análisis demuestra algo precioso: **la elegancia y la simplicidad matemática de una expresión se traducen directamente en eficiencia a bajo nivel en la computación.** Escribir menos código y más directo suele ser el camino óptimo.

¿Ves con claridad cómo la máquina virtual se ahorró el "salto lúdico" gracias a tu segunda estructura? Si es así, ¡estamos listos para el **Módulo 4: Gestión de memoria y el Recolector de Basura**!

# Ahora si el módulo 4   

El Módulo 4 es uno de los más importantes para entender el rendimiento de Python a gran escala. Como matemático, estás muy familiarizado con la **teoría de grafos** (vértices, aristas, caminos y ciclos). 

Te alegrará saber que la gestión de memoria en Python se rige exactamente bajo estas estructuras matemáticas.

---



## Módulo 4: Gestión de Memoria y el Recolector de Basura (Garbage Collector)



A diferencia de lenguajes como C o C++ (donde el programador debe reservar y liberar la memoria manualmente con comandos como `malloc` y `free`), Python gestiona la memoria de forma automática. 

Para lograrlo, utiliza un sistema de dos capas:

1. **Conteo de Referencias (Reference Counting):** La primera línea de defensa (rápida y en tiempo real).
2. **El Recolector de Basura Generacional (Garbage Collector - GC):** La segunda línea de defensa (enfocada en resolver problemas que el conteo de referencias no puede solucionar).

---



### 1. El Conteo de Referencias: El "Grado de Entrada" de un Vértice

En CPython, cada variable es un puntero que apunta a un objeto en memoria.

> **Perspectiva Matemática:** Imagina la memoria de Python como un **grafo dirigido** donde los vértices (nodos) son los objetos en memoria y las aristas (flechas) son las referencias de las variables hacia esos objetos. El conteo de referencias es equivalente al **grado de entrada** (in-degree) de cada nodo.

Cada vez que asignas un objeto a una nueva variable, su grado de entrada aumenta en 1. Si la variable sale de su alcance (por ejemplo, al terminar una función) o se reasigna, el grado de entrada disminuye en 1.

$$\text{Si } \operatorname{in-degree}(\text{objeto}) = 0 \implies \text{El objeto se destruye inmediatamente y se libera su memoria.}$$

---



### 2. El Problema de las Referencias Cíclicas (Componentes Fuertemente Conexos)

El conteo de referencias tiene un punto ciego catastrófico: **los ciclos**.

Imagina que el objeto $A$ apunta al objeto $B$, y el objeto $B$ apunta al objeto $A$. Aunque borres las variables externas que apuntaban a ellos, sus grados de entrada individuales nunca llegarán a cero porque se apuntan mutuamente.

```
[ Variable Externa ] (Se elimina)
         │
         ▼
     [ Objeto A ] ◄──────┐  (In-degree = 1)
         │               │
         ▼               │
     [ Objeto B ] ───────┘  (In-degree = 1)

```



Ambos objetos están flotando en la memoria, inaccesibles para tu programa, pero consumiendo recursos. Esto se conoce como una **fuga de memoria (memory leak)**.



Aquí es donde entra el **Garbage Collector (GC)** de Python. 

Su trabajo es buscar periódicamente de forma activa componentes fuertemente conexos (ciclos) en el grafo de memoria que hayan quedado aislados del espacio de nombres principal, y destruirlos de raíz.

---



## Ejemplos de Aplicación Práctica en tu Jupyter o VS Code

Ejecutemos los siguientes experimentos para interactuar directamente con la memoria de CPython.



### Ejemplo 1: Observando el conteo de referencias en tiempo real

Utilizaremos el módulo `sys` para inspeccionar cuántas flechas apuntan a un objeto.



In [22]:
import sys

# Creamos una lista en memoria
lista_a = [1, 2, 3]

# Obtenemos su conteo de referencias
# Nota: sys.getrefcount() suma 1 temporalmente porque la función misma
# necesita crear una referencia para inspeccionarla.
print("Referencias iniciales:", sys.getrefcount(lista_a) - 1) 

# Creamos una segunda variable que apunta a la misma lista (creamos otra arista en el grafo)
lista_b = lista_a
print("Referencias tras asignar lista_b:", sys.getrefcount(lista_a) - 1)

# Eliminamos una variable
del lista_b
print("Referencias tras eliminar lista_b:", sys.getrefcount(lista_a) - 1)


Referencias iniciales: 1
Referencias tras asignar lista_b: 2
Referencias tras eliminar lista_b: 1



### Ejemplo 2: Creando una fuga de memoria (Referencia Cíclica)

Vamos a modelar una estructura matemática común en Python: un **nodo de un grafo** o un elemento de una lista enlazada doble, donde dos objetos se apuntan mutuamente. Veremos cómo el conteo de referencias falla y cómo el recolector de basura tiene que intervenir.



In [23]:
import sys
import gc

# Nos aseguramos de que el Recolector de Basura esté activado
gc.enable()

class Nodo:
    def __init__(self, nombre):
        self.nombre = nombre
        self.vecino = None
    def __del__(self):
        # Este método se ejecuta cuando el objeto es destruido de la memoria
        print(f"-> {self.nombre} ha sido eliminado de la memoria")

# 1. Creamos un entorno controlado usando una función
def crear_ciclo():
    nodo_1 = Nodo("Nodo A")
    nodo_2 = Nodo("Nodo B")
    
    # Creamos la referencia cíclica (A apunta a B, B apunta a A)
    nodo_1.vecino = nodo_2
    nodo_2.vecino = nodo_1
    
    print("Saliendo de la función crear_ciclo()...")
    # Al salir de aquí, las variables 'nodo_1' y 'nodo_2' dejan de existir localmente.

crear_ciclo()
print("La función terminó.")


Saliendo de la función crear_ciclo()...
La función terminó.



**Si ejecutas esto en tu celda, notarás algo impactante:**
Al terminar la función y decir `"La función terminó."`, ¡no aparece el mensaje de destrucción de los nodos! Siguen atrapados en la memoria porque se apuntan mutuamente (su conteo de referencias interno es $1$).



Para liberarlos, debemos forzar al Recolector de Basura a realizar una inspección de grafos:



In [ ]:
print("Forzando la recolección de basura...")
objetos_recogidos = gc.collect() # Analiza el grafo de memoria buscando ciclos aislados
print(f"Objetos destruidos por el GC: {objetos_recogidos}")



Inmediatamente verás aparecer los mensajes `-> Nodo A ha sido eliminado...` y `-> Nodo B ha sido eliminado...`. El algoritmo del GC detectó el ciclo cerrado y rompió el nudo.

---



### Ejemplo 3: El truco de las Referencias Débiles (`weakref`)

Si estás construyendo estructuras de datos complejas (como árboles de decisión, redes de neuronas, o grafos personalizados para física/matemática) y no quieres sobrecargar al Garbage Collector con búsquedas de ciclos, puedes usar **referencias débiles**.



Una referencia débil apunta a un objeto pero **no incrementa su conteo de referencias**. Si el objeto se destruye en otro lado, la referencia débil simplemente pasa a apuntar a `None` de forma segura.


In [ ]:
import weakref

class ObjetoMatematico:
    def __init__(self, valor):
        self.valor = valor

# Creamos un objeto real
obj = ObjetoMatematico(3.14159)

# Creamos una referencia débil hacia él
ref_debil = weakref.ref(obj)

# Accedemos al objeto a través de la referencia débil
print("Valor recuperado:", ref_debil().valor)

# Eliminamos la única referencia real
del obj

# Intentamos acceder de nuevo
print("¿Sigue existiendo el objeto?:", ref_debil())
# Imprimirá None, porque el objeto se destruyó inmediatamente al eliminar su única referencia fuerte.



## Tu turno de analizar:

Las bases de datos y las grandes matrices en NumPy o Pandas consumen muchísima memoria RAM.

Pensando en la charla de la PyCon:
Si estás procesando una matriz gigantesca de datos meteorológicos y de repente tu script se ralentiza o se queda sin memoria, ¿crees que es mejor confiar en el recolector de basura automático de Python (`gc`), o es preferible estructurar tu código para que el conteo de referencias (`del` + scope de funciones) actúe de inmediato?

¿Qué opinas basándote en la velocidad con la que trabaja cada uno de estos dos sistemas?

# Finalmente sigamos con el módulo 5  

¡Totalmente! Vamos al **Módulo 5**, el gran cierre de esta ruta de aprendizaje.

Este es el tema estrella, el más "caliente" y debatido en la comunidad global de Python. Como matemático y académico, este módulo te resultará apasionante porque se trata de cómo resolver un problema de **concurrencia y sincronización en sistemas distribuidos** (en este caso, los múltiples núcleos de tu propio procesador).

---

## Módulo 5: Concurrencia y el fin del GIL (El Santo Grial de Python)

Para entender qué es el GIL y por qué su eliminación es una revolución, primero debemos definir dos conceptos que la gente suele confundir, pero que matemáticamente son muy diferentes:

* **Concurrencia (Estructura):** Es la propiedad de un programa de poder ser dividido en partes que pueden ejecutarse en un orden parcial o desordenado sin afectar el resultado final. Es un problema de **diseño**.
* **Paralelismo (Ejecución):** Es la ejecución física y simultánea de múltiples tareas en diferentes recursos de hardware (por ejemplo, el núcleo 1 de tu CPU hace el cálculo $A$ mientras el núcleo 2 hace el cálculo $B$). Es un problema de **infraestructura**.

---

### 1. El gran villano: El GIL (Global Interpreter Lock)

Históricamente, CPython (el intérprete estándar escrito en C) **no fue diseñado para ser seguro con hilos (thread-safe)**.

Si dos hilos que corren al mismo tiempo intentan modificar el conteo de referencias de un mismo objeto en memoria al mismo segundo, podría ocurrir una **condición de carrera (race condition)**. El conteo de referencias podría corromperse, borrando de la memoria un objeto que aún se necesita o dejando basura para siempre.

Para evitar este caos de memoria, los creadores de Python implementaron una solución drástica: el **GIL**.

> **Definición de GIL:** Es un "candado" (lock) único a nivel global que debe adquirir cualquier hilo para poder interactuar con la máquina virtual de Python.

```
                       [ MÚLTIPLES HILOS ]
                       ┌───┐  ┌───┐  ┌───┐
                       │H1 │  │H2 │  │H3 │
                       └───┘  └───┘  └───┘
                         │      │      │
                         ▼      ▼      ▼
                       ┌─────────────────┐
                       │     EL GIL      │ ◄─── (Solo deja pasar a UN hilo a la vez)
                       └─────────────────┘
                                │
                                ▼
                       ┌─────────────────┐
                       │ Máquina Virtual │
                       └─────────────────┘

```

**La consecuencia matemática/computacional:** Aunque tu computador tenga 16 núcleos de CPU de última generación, si usas hilos (*threads*) en Python para tareas que consumen procesador (como cálculos matemáticos puros), **Python solo usará 1 núcleo a la vez**. Los otros 15 núcleos se quedarán de brazos cruzados esperando que el GIL libere el único carril de ejecución.

---

### 2. La Revolución: PEP 703 y "Free-threaded Python"

Daniel Arango cerrará su charla explicando cómo el equipo de desarrollo de Python logró finalmente remover el GIL (un esfuerzo titánico que tomó años de ingeniería). Esta versión sin GIL (conocida como *Free-threaded Python*) ya es una realidad opcional y permite que Python corra de forma **verdaderamente paralela en múltiples núcleos**.

---

## Ejemplos Ilustrativos y de Aplicación Práctica

Para entender el impacto del GIL y cómo se comporta el cómputo en Python, veamos estos ejemplos prácticos que puedes analizar en VS Code.

### Ejemplo 1: El cuello de botella del GIL en tareas matemáticas (CPU-bound)

Vamos a simular una tarea pesada: calcular la suma de los números hasta 50 millones. Compararemos cuánto tarda un solo hilo frente a dividir el trabajo en dos hilos paralelos.

```python
import time
import threading

LIMITE = 50_000_000

def contar(n):
    total = 0
    for i in range(n):
        total += i
    return total

# --- EXPERIMENTO 1: UN SOLO HILO ---
inicio = time.time()
contar(LIMITE)
fin = time.time()
print(f"Un solo hilo tardó: {fin - inicio:.4f} segundos")

# --- EXPERIMENTO 2: DOS HILOS EN PARALELO ---
# Dividimos el trabajo en dos hilos de 25 millones cada uno
hilo1 = threading.Thread(target=contar, args=(LIMITE // 2,))
hilo2 = threading.Thread(target=contar, args=(LIMITE // 2,))

inicio = time.time()
hilo1.start()
hilo2.start()

# Esperamos a que ambos hilos terminen
hilo1.join()
hilo2.join()
fin = time.time()
print(f"Dos hilos en paralelo tardaron: {fin - inicio:.4f} segundos")

```

#### Lo que observarás en tu entorno tradicional (con GIL):

Te llevarás la sorpresa de que **usar dos hilos tarda prácticamente lo mismo (o incluso más)** que usar un solo hilo. Esto ocurre porque el GIL obligó a los dos hilos a turnarse constantemente en el mismo núcleo de la CPU, agregando además el costo de tiempo de estar alternando el candado de uno a otro.

---

### Ejemplo 2: El comportamiento ideal del multiprocesamiento (La alternativa matemática)

Dado que el GIL bloquea los hilos dentro del *mismo* intérprete, la solución histórica en Python para paralelizar matemáticas pesadas ha sido usar **procesos independientes** en lugar de hilos.

Cada proceso tiene su propia memoria y su propio intérprete de Python (con su propio GIL individual), por lo que sí pueden correr en núcleos separados de tu CPU al mismo tiempo.



In [ ]:
import time
from multiprocessing import Process

def contar(n):
    total = 0
    for i in range(n):
        total += i
    return total

if __name__ == "__main__":
    LIMITE = 50_000_000

    # Creamos dos procesos independientes
    proceso1 = Process(target=contar, args=(LIMITE // 2,))
    proceso2 = Process(target=contar, args=(LIMITE // 2,))

    inicio = time.time()
    proceso1.start()
    proceso2.start()

    proceso1.join()
    proceso2.join()
    fin = time.time()
    
    print(f"Usando Multiprocesamiento (2 núcleos reales) tardó: {fin - inicio:.4f} segundos")



## Lo que observarás:

Aquí verás una reducción drástica del tiempo (cercana al 40-50% si tu CPU tiene más de un núcleo). Al ser procesos separados, la CPU de verdad distribuyó la carga de trabajo de forma paralela.

---



### Ejemplo 3: ¿Qué cambia en Python sin el GIL? (El futuro)

En la charla, Daniel explicará que al remover el GIL, el intérprete debe proteger la memoria usando **candados internos a nivel de objeto** (mucho más granulares) en lugar de un único gran candado global.

Esto beneficia enormemente a:

1. **Librerías de Inteligencia Artificial y Redes Neuronales** (como PyTorch y TensorFlow), que podrán mover tensores y matrices en memoria directamente usando hilos de Python a velocidades extremas.
2. **Cómputo Científico en NumPy:** El procesamiento paralelo de matrices será nativo sin la necesidad de liberar manualmente el GIL en el código de C de la librería.

---



## Cierre de la preparación para la PyCon



¡Hemos recorrido todo el camino! Desde entender que Python es solo una especificación abstracta, pasando por las estructuras arbóreas del AST, la optimización lineal del bytecode, la teoría de grafos en la recolección de basura, hasta llegar al paralelismo puro sin el GIL.



Con esta base conceptual sólida y la traducción que analizamos de la charla, estás **más que listo** para sentarte en la primera fila de la conferencia de Daniel Arango en la PyCon Colombia. 

Podrás seguir cada uno de sus diagramas de AST, entenderás perfectamente cuando hable de la transición de LL(1) a PEG, sabrás qué significa que el desensamblador muestre bytecode sin el GIL y disfrutarás de su anécdota sobre el intento de acelerar el intérprete.

# Posible ponencia de Daniel Arango  

Basándonos en los temas técnicos avanzados de su resumen de presentación y en su biografía (donde menciona batallas contra "ASTs idénticos" y un intento fallido de acelerar el propio intérprete de Python), este es un **ejemplo muy posible del escenario real, el código y la anécdota** que Daniel Arango podría presentar en el escenario de la PyCon Colombia:

---



## El escenario de la ponencia: "El día que creí haber hackeado a CPython (pero olvidé cómo funciona el compilador)"

Daniel subirá al escenario y probablemente planteará un acertijo lógico-matemático basado en la optimización de código. Te mostrará un ejemplo de cómo funciona el compilador interno de CPython utilizando una técnica real llamada **Peephole Optimization** (una optimización que hace el compilador a nivel de bytecode justo antes de ejecutar el programa).

### El Acertijo del Speaker

Imagina que quieres escribir una función matemática en Python que convierta días a segundos. Daniel nos presentará dos formas de escribirla:

```python


In [26]:
# Opción A: Descriptiva y lógica (para humanos)
def segundos_en_un_dia_A():
    return 24 * 60 * 60

# Opción B: Con el cálculo ya resuelto (para ahorrarle trabajo a la máquina)
def segundos_en_un_dia_B():
    return 86400




Cualquier programador junior pensaría: *"La opción B debe ser más rápida porque la CPU no tiene que multiplicar tres números cada vez que llamo a la función"*.



De hecho, si intentas optimizar el intérprete de Python para que sea más veloz, podrías pensar en escribir un parche en C para CPython que detecte estas operaciones aritméticas constantes y las resuelva en caliente.

### La revelación del AST y el Bytecode (La "Batalla")

Daniel mostrará en la pantalla gigante de la PyCon qué pasa bajo el capó cuando desensamblamos ambas funciones usando `dis`:



In [27]:
import dis

print("--- Opción A (24 * 60 * 60) ---")
dis.dis(segundos_en_un_dia_A)

print("\n--- Opción B (86400) ---")
dis.dis(segundos_en_un_dia_B)


--- Opción A (24 * 60 * 60) ---
  2           RESUME                   0

  3           RETURN_CONST             1 (86400)

--- Opción B (86400) ---
  6           RESUME                   0

  7           RETURN_CONST             1 (86400)



Para sorpresa de muchos en la audiencia, el bytecode de **ambas** funciones es **exactamente el mismo**:

```text
# Bytecode de la Opción A:               # Bytecode de la Opción B:
  2           0 LOAD_CONST               1 (86400)    2           0 LOAD_CONST               1 (86400)
              2 RETURN_VALUE                                      2 RETURN_VALUE

```



#### ¿Por qué los ASTs y el Bytecode terminaron siendo idénticos?

Aquí es donde Daniel explicará la magia del compilador:
Durante la fase de análisis, el parser construye el AST. El compilador de Python ve que en la Opción A hay un nodo de operación binaria (`BinOp`) cuyas hojas son puras constantes numéricas (`24`, `60`, `60`).

Como esos números nunca van a cambiar en tiempo de ejecución, el compilador realiza un proceso llamado **Constant Folding** (plegado de constantes). Reemplaza todo ese subárbol del AST por un solo nodo de constante con el valor precalculado: `86400`.

Por lo tanto, ambas opciones generan un **AST final y un bytecode idénticos**. El desarrollador no gana ni un microsegundo de velocidad escribiendo `86400` directamente.

---



### La anécdota y la "derrota" técnica de Daniel

Aquí viene la parte divertida e inspiradora de la charla que Daniel adelanta en su sinopsis:

> *"Creí haber descubierto una manera de acelerar el propio intérprete de Python..."*

Daniel nos contará cómo intentó escribir un algoritmo en C para el motor de CPython que hiciera optimizaciones masivas en expresiones algebraicas complejas dentro del AST (por ejemplo, simplificar $x + 0 \to x$ o $y \times 1 \to y$). Él pensó que al reducir dinámicamente estas expresiones en el árbol sintáctico, la máquina virtual ejecutaría menos bytecode y el intérprete volaría.

**¿Pero... qué pasó?**
Al implementar su idea y medir los tiempos de ejecución, descubrió que **su optimizador ralentizaba a Python en lugar de acelerarlo**.

El costo computacional (en tiempo de CPU) de recorrer recursivamente todo el árbol de sintaxis buscando patrones algebraicos para simplificar era **mucho mayor** que el pequeño beneficio de ahorrarse un par de instrucciones básicas de bytecode (`BINARY_OP`) en la máquina virtual. El compilador tardaba más tiempo analizando el código que lo que la máquina virtual tardaba en ejecutarlo de la forma "no optimizada".

---

### Qué te llevarás de esta sesión en la PyCon:

Este ejemplo hipotético ilustra a la perfección el núcleo de su ponencia:

* Python no es un simple script que corre línea a línea; tiene un compilador inteligente intermedio.
* No intentes "ganarle" al compilador de Python haciendo optimizaciones manuales extrañas en tus fórmulas matemáticas (como precalcular valores constantes o evitar paréntesis).
* Comprender el AST te ayuda a entender que lo que tú escribes en VS Code no es necesariamente lo que la máquina virtual termina ejecutando.

# Diseño de curso de inglés para participar en el congreso  



Para un profesional con tu formación matemática y un perfil analítico estructurado, aprender inglés de oído no suele ser el camino más eficiente. 

Es mucho mejor enfocar el idioma como un **sistema lógico de mapeo**: aprender los términos precisos, las estructuras gramaticales de causa-efecto que usan los tecnólogos y los patrones de entonación.



Dado que la PyCon Colombia 2026 se celebra en Medellín, este plan de entrenamiento intensivo está diseñado para que en un tiempo récord puedas decodificar tanto el soporte visual (las diapositivas) como el inglés hablado de Daniel Arango Sohm.

---



# Programa de Inglés Técnico para la PyCon

```
             ┌────────────────────────────────────────────────────────┐
             │       PROGRAMA INTENSIVO: INGLÉS PARA INGENIERÍA       │
             └────────────────────────────────────────────────────────┘
                                          │
         ┌────────────────────────────────┼───────────────────────────────┐
         ▼                                ▼                               ▼
   [ Módulo A ]                     [ Módulo B ]                    [ Módulo C ]
  Glosario Lógico                Patrones Sintácticos             Entrenamiento de Oído
  Mapeo de términos              "If-Then", "Under the hood"      Ritmo de hablante nativo
  técnicos del intérprete        y conectores de flujo            de español en entornos IT

```



# Módulo A: El Diccionario Lógico (Mapeo de Términos)

En lugar de memorizar miles de palabras en inglés, necesitas dominar este subconjunto cerrado de términos que aparecerán sí o sí en las diapositivas de la charla de Daniel.

| Término en Inglés | Pronunciación Fonética (Aproximada) | Significado Técnico en el Contexto |
| --- | --- | --- |
| **Under the hood** | *án-der de jud* | "Bajo el capó" (cómo funciona internamente el código). |
| **Garbage Collector (GC)** | *gár-bich co-léc-tor* | Recolector de basura (limpieza de memoria). |
| **Reference Counting** | *ré-fer-ens cáun-ting* | Conteo de referencias. |
| **Abstract Syntax Tree (AST)** | *áb-stract sín-tax trii* | Árbol de sintaxis abstracta (la estructura de grafo). |
| **Peephole Optimization** | *píip-jol op-ti-mai-zéi-shon* | Optimización en ventana/pequeña (cambiar $2*3 \to 6$). |
| **Thread-safe** | *zred-séif* | Seguro para hilos (sin colisiones en memoria). |
| **Bottleneck** | *bó-tel-nec* | Cuello de botella (la parte lenta del algoritmo). |
| **To bypass** | *tu bái-pas* | Esquivar / Saltarse (ej. esquivar el GIL). |
| **To deprecate** | *tu dé-pre-keit* | Descontinuar / Volver obsoleto (ej. el parser LL(1)). |

---



# Módulo B: Patrones Sintácticos de Causa-Efecto y Argumentación



Los científicos y programadores utilizan estructuras muy predecibles cuando explican sistemas lógicos. 

Si dominas estas 4 estructuras, entenderás de inmediato las transiciones y argumentos de la ponencia:



### 1. El patrón de "Desafío a la lógica común"

Los expositores suelen usar esto para captar tu atención antes de revelar un secreto de bajo nivel.

* **La estructura en inglés:** *"We tend to believe that `[Hipótesis]`... however, in reality, `[Hecho real]`."*
* **Traducción:** *"Solemos creer que `[Hipótesis]`... sin embargo, en realidad, `[Hecho real]`."*
* **Ejemplo de la charla:** *"We tend to believe that Python is just interpreted; however, in reality, CPython compiles it to bytecode first."*



### 2. El patrón de Consecuencia Inmediata (Condicionales)

* **La estructura en inglés:** *"If we `[Acción]`, the interpreter has to `[Reacción]`..."*
* **Traducción:** *"Si hacemos `[Acción]`, el intérprete tiene que `[Reacción]`..."*
* **Ejemplo de la charla:** *"If we use threads for CPU-bound tasks, the GIL will block parallel execution."*



### 3. El patrón del "Pero..." (La anécdota del speaker)

Daniel hablará de cuando creyó acelerar el intérprete, pero algo falló. Atento a este conector:

* **La estructura en inglés:** *"I managed to `[Logro]`, but it turned out to `[Consecuencia negativa]`..."*
* **Traducción:** *"Logré `[Logro]`, pero resultó que `[Consecuencia negativa]`..."*
* **Ejemplo de la charla:** *"I managed to optimize the AST nodes, but it turned out to slow down the parser because of the recursion overhead."*

---



## Módulo C: El Oído y el Ritmo en Charlas de IT

Daniel Arango es colombiano (con herencia alemana y rusa). Esto es una **excelente noticia para tu oído**:

1. **La estructura del acento:** Los hablantes nativos de español que dan charlas técnicas en inglés suelen tener una pronunciación muy clara, con vocales bien definidas y un ritmo pausado. No vas a enfrentarte a los modismos o la rapidez de un estadounidense nativo.
2. **Mapeo de "Spanglish" técnico:** Muchas palabras se asimilan directamente. Cuando Daniel diga en inglés *"The collector scans the graph..."*, tu cerebro matemático lo traducirá casi instantáneamente a *"El colector escanea el grafo..."*.

---



# Tu Entrenamiento Práctico para esta Semana

Para afinar el oído antes del evento, haz este ejercicio de 15 minutos en tu computador:

1. **Mira un video técnico corto:** Busca en YouTube un video de 5 minutos sobre [*"Python AST Tutorial"](https://www.youtube.com/watch?v=tM_S-pa4xDk&list=PLw5h0DiJ-9PCWamtYU7X220dlBSU94BdD)* o *"CPython Bytecode"* en inglés.
2. **Paso 1 (Con subtítulos en inglés):** Míralo una vez prestando atención a cómo se escriben y suenan las palabras del **Módulo A**.
3. **Paso 2 (Sin subtítulos):** Míralo por segunda vez intentando capturar las estructuras de causa-efecto del **Módulo B**.



Al entrenar tu cerebro para buscar estas "piezas lógicas" en lugar de intentar traducir palabra por palabra, podrás sentarte en la charla técnica avanzada, entender las diapositivas con confianza y captar la esencia de la ponencia sin estrés.

En este contexto de arquitectura de Python, la traducción más precisa y natural de: "If we use threads for CPU-bound tasks, the GIL will block parallel execution." es:

> **"Si usamos hilos para tareas intensivas en CPU, el GIL bloqueará la ejecución en paralelo".**

---



# Desglose de los términos clave para que brilles en la conferencia:



* **"threads" $\to$ "hilos"**: En español técnico se prefiere casi siempre el término "hilos" (abreviatura de *hilos de ejecución*) sobre "subprocesos" o "procesos ligeros", ya que "procesos" (*processes*) es un concepto de sistema operativo completamente diferente en Python.


* **"CPU-bound tasks" $\to$ "tareas intensivas en CPU"**: Este es un término fundamental. En computación, una tarea es *CPU-bound* (ligada o limitada por la CPU) cuando su velocidad depende puramente de la potencia de procesamiento matemático/lógico de tu procesador (como entrenar un modelo o calcular decimales de $\pi$), a diferencia de las tareas *I/O-bound* (limitadas por entrada/salida), que dependen de la velocidad de tu internet o tu disco duro.


* **"parallel execution" $\to$ "ejecución en paralelo"**: Significa que varios núcleos físicos de tu procesador computen al mismo millonésimo de segundo.